# Independent Confirmation: Concept Probe (16 targets, new descriptions)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1112_qwen_probe_independent_confirmation/qwen_probe_independent_confirmation_16targets.ipynb)

**Objective:** Confirm whether the Qwen X-vs-Y Probes selected in the completed v2 exploratory screen retain linear discrimination when tested on **8 newly authored semantic fact families per concept**. The earlier screening data are read-only and **the held-out texts are not used for model fitting, distance selection, or comparator selection**.

- Primary candidates: 12 exploratory stable targets; secondary: car, elephant, scorpion, train (4 promising but incomplete targets).
- 23 distinct concepts including comparators; 8 independently authored statements per concept (184 new Qwen prompts).
- The original 8 Wikipedia-derived *training* families per concept, baseline weights of the model and primary layers/position are unchanged. **Probes are re-fitted solely on the frozen original training vectors**, because individual weights were not exported in v2.
- **Frozen comparator choices** come exclusively from the old v2 result `target_candidates.csv`; only the previously selected promising comparisons are tested. This is a prospective confirmation of the shortlist, *not* a fresh 100-concept screen.
- Evaluate L22, L23, L24 AUC, training leave-one-fact-out sensitivity, training-label permutation controls; report an FDR-adjusted sensitivity analysis as well as the original exploratory thresholds.
- Only completed results are published. Large, immutable ZIP checkpoint bundles are saved under the run-specific GitHub path to support safe resumption.

**Limitations:** New statements are AI-authored research stimuli and must be independently reviewed for factual correctness, concept ambiguity and overlap before any confirmatory thesis claim. Their creation is temporally separate from v2 evaluation results, but they are not independent human annotations. This still does not measure absolute activation or causal suppression.

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1,<2" "huggingface-hub>=0.30,<1" "scikit-learn==1.6.1" pandas numpy matplotlib requests tqdm scipy tabulate statsmodels

In [ ]:
import os, io, re, json, math, base64, hashlib, random, zipfile, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import numpy as np, pandas as pd, requests, torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from google.colab import userdata

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
REPO='ryouy/GraduationThesis'
SOURCE='1111_qwen_probe_100concepts_v2/results/20261112_qwen100_richer_dedup_v2'
RUN_ID='20261112_newfacts_16targets_v1'
DEST='1112_qwen_probe_independent_confirmation/results/'+RUN_ID
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION='7ae557604adf67be50417f59c2c2f167def9a775'
LAYERS=[22,23,24]
POSITION='assistant_start'
BATCH_SIZE=48
GROUP_BATCHES=3
WORK=Path('/content/qwen_confirmation')/RUN_ID
CACHE=WORK/'cache'; CACHE.mkdir(parents=True,exist_ok=True)
OUT=WORK/'results'; OUT.mkdir(parents=True,exist_ok=True)
try: TOKEN=userdata.get('GITHUB_RESULTS_TOKEN')
except Exception: TOKEN=None
assert TOKEN, 'Add GITHUB_RESULTS_TOKEN to Colab Secrets and grant notebook access.'
s=requests.Session();s.headers.update({'Authorization':f'Bearer {TOKEN}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'})
API=f'https://api.github.com/repos/{REPO}/contents/'
def meta(path):
    r=s.get(API+path,timeout=90)
    if r.status_code==404:return None
    r.raise_for_status();return r.json()
def get_bytes(path):
    m=meta(path)
    if m is None:return None
    # The GitHub content API sometimes truncates larger base64; use authenticated download URL.
    r=s.get(m['download_url'],timeout=240);r.raise_for_status();return r.content
def save_bytes(path,blob,message):
    gitsha=hashlib.sha1(b'blob '+str(len(blob)).encode()+b'\0'+blob).hexdigest()
    prev=meta(path)
    if prev is not None:
        if prev['sha']!=gitsha:
            raise RuntimeError('Existing GitHub artifact differs. Preserve it and choose a NEW RUN_ID: '+path)
        print('Already verified:',path);return
    r=s.put(API+path,json={'branch':'main','message':message,'content':base64.b64encode(blob).decode()},timeout=240)
    r.raise_for_status();print('Saved:',path)
def text_csv(path):
    b=get_bytes(path)
    assert b is not None, path
    return pd.read_csv(io.BytesIO(b),keep_default_na=False)


## 1. Freeze the old candidate selections and original training vectors

These GitHub resources are **read-only**: the old `manifest.json`, `stimulus_design.csv`, `target_candidates.csv`, `candidate_comparators_frozen.csv`, and `checkpoints/`. The original model revision and 8 training facts per concept are unchanged. We select comparator names **without using any newly generated AUC**.

In [ ]:
old_manifest=json.loads(get_bytes(SOURCE+'/manifest.json'))
old_stim_raw=get_bytes(SOURCE+'/stimulus_design.csv')
assert old_stim_raw is not None
old_stim=pd.read_csv(io.BytesIO(old_stim_raw),keep_default_na=False)
old_targets=text_csv(SOURCE+'/target_candidates.csv')
old_choices=text_csv(SOURCE+'/candidate_comparators_frozen.csv')
assert old_manifest['status']=='COMPLETED_EXPLORATORY_SCREEN'
assert old_manifest['model']==MODEL_ID and old_manifest['revision']==MODEL_REVISION
assert old_manifest['layers']==LAYERS and old_manifest['position']==POSITION
assert old_manifest['train_families']==8 and old_manifest['eval_families']==8
assert len(old_stim)==old_manifest['prompts']==1104
assert hashlib.sha256(old_stim_raw).hexdigest()==old_manifest['prompt_hash'], 'Source prompt identity mismatch.'

PRIMARY=['bus','motorcycle','bicycle','boat','butterfly','dragonfly','jellyfish','submarine','ant','moth','starfish','wolf']
SECONDARY=['car','elephant','scorpion','train']
TARGETS=PRIMARY+SECONDARY
assert set(TARGETS).issubset(set(old_targets.target))
# Competing Ys are frozen from the PREVIOUS run's stable-comparator list.
# This code never uses the new evaluation to add/remove pairs.
sel=[]
for x in TARGETS:
    rec=old_targets.set_index('target').loc[x]
    comparators=[v.strip() for v in rec.comparators.split(',') if v.strip()]
    assert len(comparators)>=2, f'{x}: fewer than 2 pre-frozen comparators'
    for y in comparators:
        match=old_choices[(old_choices.target==x)&(old_choices.comparator==y)]
        assert len(match)==1,(x,y)
        m=match.iloc[0]
        sel.append({'target':x,'comparator':y,'band':m.band,'cosine_distance':float(m.cosine_distance),'tier':'primary' if x in PRIMARY else 'secondary'})
PAIRS=pd.DataFrame(sel)
NAMES=sorted(set(PAIRS.target)|set(PAIRS.comparator))
assert len(NAMES)>=16
print('Frozen target concepts:',len(TARGETS),'different concepts including comparators:',len(NAMES),'directed comparisons:',len(PAIRS))
print('Distances already selected:',PAIRS.band.value_counts().to_dict())
PAIRS.to_csv(OUT/'frozen_comparators.csv',index=False)


## 2. Entirely new authored semantic facts

The following descriptions were authored **after** the prior screening and are independent *input strings* (not copied from Wikipedia). There are eight distinct fact families per concept, no repeated wrappers, and no explicit concept label inside its own text. These are proposed stimuli, **not human-validated ground truth**. The notebook publishes a lexical-overlap audit with the older 16 facts per concept for manual review.

*Do not edit these statements while keeping the run ID.*

In [ ]:
FRESH_FACTS = {'bus': ['Commuters line up at a curbside stop before entering through folding doors.', 'The driver checks a large mirror before pulling away from the passenger shelter.', 'An electronic destination sign changes when the vehicle begins another route.', 'Wheelchair users can board using a ramp lowered near the pavement.', 'Passengers sometimes press a bell to request the following stop.', 'Public transport agencies publish timetables identifying departure times for each route.', 'A long articulated vehicle uses a flexible joint to negotiate tight urban corners.', 'A fare validator near the entrance records payment when a rider boards.'], 'motorcycle': ['The rider leans the two-wheeled machine through a bend while balancing on a narrow seat.', 'Protective clothing helps reduce injury when a rider slides across asphalt.', 'A twist grip on the right handlebar controls engine power.', 'The rear wheel receives power through a chain, shaft, or drive belt.', 'A kickstand supports the machine while it is parked.', 'Riders often use hand signals or indicator lamps before changing lanes.', 'The front suspension compresses when the rider brakes sharply.', 'A passenger may ride behind the operator with feet on separate pegs.'], 'bicycle': ['Pedaling turns a chain that transfers human effort to the rear wheel.', 'A bottle cage attached to the frame allows riders to carry drinking water.', 'The saddle height can be adjusted to accommodate different leg lengths.', 'Derailleur gears move the chain among sprockets to change the effort required.', 'Many commuters fasten a U-shaped lock around the frame and a fixed post.', 'A hand-operated lever pulls a cable or hydraulic system to slow the wheels.', 'Reflectors improve visibility when the rider travels after sunset.', 'A puncture repair kit contains patches and small levers for removing a tire.'], 'boat': ['A hull displaces enough water to keep people and cargo afloat.', 'The bow rises over a small wake as the vessel accelerates across a lake.', 'A rope is secured to a cleat while the craft remains alongside the dock.', 'Shallow-draft designs can approach shorelines that deep vessels cannot reach.', 'A skipper considers wind, current, and channel markers when planning a crossing.', 'Some vessels carry oars, while others rely on sails or propellers.', 'Life jackets are stored aboard for emergencies on open water.', 'The stern swings toward the pier as a person steers through a narrow harbor.'], 'butterfly': ['Scales covering delicate wings produce patterns that can warn predators.', 'A coiled feeding tube unrolls to reach nectar deep inside a flower.', 'The larval stage consumes vegetation before forming a protective chrysalis.', 'Adults use antennae to detect odors while searching for food or mates.', 'In some species, migrants travel enormous distances between breeding areas.', 'Courtship may involve fluttering displays above sunlit patches of vegetation.', 'A resting individual holds colorful wings upright over its body.', 'Many species depend on particular plants where females deposit eggs.'], 'dragonfly': ['Two long pairs of wings allow abrupt turns and hovering above ponds.', 'Large compound eyes occupy much of the head and help track flying prey.', 'Aquatic juveniles capture small organisms with a rapidly extendable mouthpart.', 'Adults seize mosquitoes and other insects while flying at high speed.', 'A female may dip the tip of her abdomen into water to place eggs.', 'The slender abdomen can remain almost horizontal when the insect perches.', 'Some species patrol the same stretch of stream repeatedly as a territory.', 'A newly emerged adult expands its wings beside an empty aquatic exoskeleton.'], 'jellyfish': ['A gelatinous bell contracts rhythmically and pushes the animal through seawater.', 'Trailing tentacles can contain cells that inject venom into nearby prey.', 'Many species drift with currents because they cannot swim strongly against them.', 'The body lacks a rigid skeleton and is composed mostly of water.', 'Some individuals emit visible light through biological chemical reactions.', 'A ring of tissues at the bell margin helps coordinate swimming motions.', 'The free-floating stage often develops from a stationary polyp earlier in life.', 'Coastal blooms can become dense enough to interfere with fishing nets.'], 'submarine': ['Ballast tanks fill with seawater to make the vessel sink below the surface.', 'An onboard system removes carbon dioxide to support a submerged crew.', 'The pressure-resistant inner hull protects personnel while operating at depth.', 'Sonar instruments locate obstacles using sound traveling through water.', 'A periscope may allow observation while most of the vessel remains hidden.', 'The craft can surface by blowing compressed air into ballast tanks.', 'Crew members navigate underwater without directly seeing the surrounding seafloor.', 'Some long-range military vessels use a nuclear reactor for sustained propulsion.'], 'ant': ['Workers carry fragments of food along chemically marked paths to a shared nest.', "A queen's principal reproductive role is to lay eggs for the colony.", 'Some species cultivate fungus gardens in underground chambers.', 'Individuals recognize nestmates through chemical cues on their outer bodies.', 'A narrow waist connects the thorax to the segmented abdomen.', 'Many workers cooperate to move objects too heavy for one individual.', 'Certain species defend their colonies using stings or irritating secretions.', 'Scouts recruit nestmates when they discover a valuable food source.'], 'moth': ['Many adults orient toward light sources during nighttime flight.', 'Feathery antennae in some males are highly sensitive to the scent of females.', 'A caterpillar may spin silk around itself before undergoing metamorphosis.', 'Wings often have muted colors that help conceal the insect against tree bark.', 'Some adults feed on flower nectar while others do not feed at all.', 'The insect typically rests with wings folded around its body like a tent.', 'Certain larvae are agricultural pests because they eat stored grain or crops.', 'Some species use ultrasound-sensitive organs to evade hunting bats.'], 'starfish': ['Tube feet beneath each arm grip surfaces using a hydraulic water system.', 'Many individuals can regenerate part of a lost limb after injury.', 'The mouth is usually located on the underside of the central disc.', 'Some species pry open bivalve shells and digest prey using an everted stomach.', 'Tiny calcified structures embedded in the skin provide support without a backbone.', 'Individuals often move slowly across rocky intertidal surfaces.', 'A light-sensitive eyespot may be present near the tip of each arm.', 'Most species have five radiating limbs, although others have many more.'], 'wolf': ['A family group communicates through long-distance howls in a forest valley.', 'Cooperative hunters may pursue hoofed animals through deep winter snow.', 'An adult can travel far from its birthplace before joining another pack.', 'Territory boundaries are sometimes signaled through scent marks.', 'Thick winter fur protects the predator during prolonged exposure to cold.', 'Young pups remain near a sheltered den while adults bring back food.', 'The animal has strong jaws and endurance suited to sustained pursuit.', 'Pack members use body posture and facial expressions during social interactions.'], 'car': ['A steering wheel turns the front road wheels while the driver sits inside an enclosed cabin.', 'Seat belts restrain occupants during sudden braking or a collision.', 'A dashboard displays speed, fuel level, or remaining battery charge.', 'A trunk behind the passenger compartment stores luggage for a trip.', "Windshield wipers clear rainfall from the driver's forward view.", 'The vehicle generally has four tires in contact with a paved road.', 'Electric versions recharge their traction batteries through a plug-in connector.', 'Rear parking sensors warn the driver when approaching an obstacle.'], 'elephant': ['A long flexible nose can lift food and draw water toward the mouth.', 'Adults cool their bodies by moving broad ears in warm weather.', 'Females and their offspring may remain together in long-lasting family groups.', 'Large tusks can be used to dig in soil or strip bark from trees.', "The animal's weight leaves wide, round impressions in soft ground.", 'A newborn calf may receive protection from several adult females.', 'Rumbling calls can carry information over distances across a savanna.', 'The herbivore spends much of the day consuming grasses, leaves, and branches.'], 'scorpion': ['A curved tail carries a venom-injecting stinger above the abdomen.', 'Pincer-like appendages grasp prey before it is subdued.', 'Many species shelter beneath stones during the heat of the day.', 'Under ultraviolet light, the outer covering may glow a blue-green color.', 'Small sensory structures on the underside detect vibrations in the ground.', 'Eight walking legs place this creature among the arachnids.', 'Some mothers carry newly born young on their backs until they molt.', 'The nocturnal hunter may wait motionless for passing insects.'], 'train': ['A locomotive pulls interconnected passenger coaches across parallel steel rails.', 'Signals along the track direct engineers when to slow or stop.', 'At a station platform, doors open only after the carriages come to rest.', 'Wheels with flanges keep the heavy vehicles aligned with the track.', 'A timetable coordinates departures along a shared rail corridor.', 'Some electric units draw power from overhead wires using a roof-mounted collector.', 'Freight consists of multiple wagons carrying containers, minerals, or grain.', 'Switches let a long set of carriages move from one rail line to another.'], 'octopus': ['Eight muscular arms bend independently to grip prey and crawl across rocks.', 'Suckers along the limbs can taste and hold onto surfaces.', 'The animal may expel a dark cloud when threatened by a predator.', 'Color-changing skin cells help the creature blend with surrounding coral.', 'A hard beak lies at the center where the arms meet.', 'Some individuals collect shells and stones to build temporary shelters.', 'Water expelled through a muscular siphon provides quick escape movement.', 'The soft-bodied hunter can squeeze through a surprisingly narrow opening.'], 'dolphin': ['A curved dorsal fin cuts the water as a social swimmer surfaces for air.', 'The animal uses clicking sounds and echoes to locate fish underwater.', 'Individuals often travel in coordinated groups called pods.', 'A horizontal tail fluke moves up and down to propel the body forward.', 'Mothers nurse their calves with milk while swimming alongside them.', 'High-pitched whistles can function as individual acoustic signatures.', 'Some populations cooperate to herd fish into compact schools.', 'The blowhole on top of the head opens briefly during a breath.'], 'whale': ['An enormous marine mammal periodically surfaces to breathe through a blowhole.', 'Some species filter microscopic prey using plates of keratin instead of teeth.', 'Low-frequency calls can propagate across large areas of the ocean.', 'A thick insulating layer of fat helps maintain warmth in cold water.', 'The horizontal tail flukes rise above the sea during a deep dive.', 'Long migrations may connect polar feeding grounds with tropical breeding waters.', 'A newborn calf swims beside its mother and drinks rich milk.', 'Vessel strikes and entanglement in fishing gear threaten some populations.'], 'bear': ['A large omnivore may spend the cold season inside a sheltered den.', 'Powerful claws turn over logs while searching for insects and other food.', 'An adult can stand upright briefly to investigate its surroundings.', 'Some species accumulate considerable fat before winter dormancy.', 'A mother guards her young closely during the first months outside the den.', 'The animal relies heavily on a keen sense of smell to locate food.', 'Depending on habitat, the diet can include berries, roots, fish, or small mammals.', 'Broad paws leave distinct tracks in muddy ground.'], 'helicopter': ['A spinning rotor produces lift that lets the aircraft rise vertically from the ground.', 'The pilot can hover over one spot while preparing a rescue operation.', 'A tail rotor on some designs prevents the fuselage from rotating uncontrollably.', 'Landing skids or wheels support the aircraft when its engine is off.', 'An overhead set of long blades sweeps through the air during takeoff.', 'Rescue crews can lower a winch to retrieve people from inaccessible terrain.', 'The cockpit controls include a collective lever that changes rotor blade pitch.', 'A small helipad provides a landing area without requiring a long runway.'], 'spider': ['Silk released from glands near the abdomen forms threads used in webs or nests.', 'Eight jointed legs allow the hunter to climb walls and vegetation.', 'Some species wait in elaborate circular traps for flying insects to become entangled.', 'Tiny sensory hairs detect vibrations passing through the silken threads.', 'Certain hunters jump directly onto prey rather than building a capture web.', 'Males may perform careful courtship movements to avoid being mistaken for prey.', 'An exoskeleton is shed periodically as the animal grows larger.', 'A pair of fangs can deliver venom into small insects during feeding.'], 'lion': ['A social group of large cats rests in shade during the hottest hours.', 'Adult males in some populations develop a thick mane around the head.', 'Females sometimes coordinate their approaches while hunting grazing herbivores.', 'A deep roar can be heard across open grassland at night.', 'Young cubs practice stalking by pouncing on siblings.', 'The predator has retractable claws and long canine teeth.', 'A pride may defend an area containing water and hunting grounds.', 'Individuals spend many daylight hours sleeping or resting.']}
assert set(FRESH_FACTS)==set(NAMES), (sorted(set(FRESH_FACTS)-set(NAMES)),sorted(set(NAMES)-set(FRESH_FACTS)))
records=[]
for name in NAMES:
    for j,fact in enumerate(FRESH_FACTS[name]):
        records.append({'concept':name,'family':j,'fact':fact,
                        'user_prompt':'Read the factual note carefully.\n\nNote: '+fact+'\n\nBriefly summarize the note.'})
NEW=pd.DataFrame(records)
assert len(NEW)==8*len(NAMES)
assert NEW.fact.nunique()==len(NEW), 'Duplicate new facts across concepts'
assert not NEW.user_prompt.duplicated().any()
OLD=old_stim[old_stim.concept.isin(NAMES)].copy()
assert not (set(NEW.fact.str.casefold())&set(OLD.fact.str.casefold())), 'New statement exactly duplicates an original statement'

# Inspect highest lexical overlap between each new fact and any previously used fact.
def tokens(text): return set(re.findall(r'[a-z]{3,}',text.casefold()))
def jaccard(a,b):
    aa,bb=tokens(a),tokens(b)
    return len(aa&bb)/max(1,len(aa|bb))
audit=[]
for row in NEW.itertuples():
    old=OLD.loc[OLD.concept==row.concept]
    matches=[(jaccard(row.fact,o.fact),o.fact,o.split) for o in old.itertuples()]
    best=max(matches,key=lambda x:x[0])
    audit.append({'concept':row.concept,'family':row.family,'maximum_old_jaccard':best[0],
                  'closest_old_statement':best[1], 'closest_old_split':best[2]})
AUDIT=pd.DataFrame(audit)
print('Maximum lexical overlap:',AUDIT.maximum_old_jaccard.max())
print(AUDIT.nlargest(12,'maximum_old_jaccard').to_string(index=False))
# Hard stop for obvious near-verbatim reuse; remaining overlap requires human inspection.
assert AUDIT.maximum_old_jaccard.max()<0.75, 'High-overlap text: revise the candidate and choose a new RUN_ID'
NEW.to_csv(OUT/'fresh_eval_design.csv',index=False)
AUDIT.to_csv(OUT/'fresh_eval_lexical_audit.csv',index=False)
print('Fresh independent evaluation facts:',len(NEW))


## 3. Recover the **original training vectors** (read-only)

This does not rerun Qwen for the training prompts. The exact previous feature checkpoints must agree with the source manifest's frozen prompt hash, batch size, and layer shape. None of the previous evaluation vectors is used for training.

In [ ]:
PREV_BATCH=64
old_layers=old_manifest['layers']
OLD_N=len(old_stim)
old_nbatch=math.ceil(OLD_N/PREV_BATCH)
OLD_FEATURES=[]
for group in tqdm(range(math.ceil(old_nbatch/4)),desc='Recover previous run features'):
    remote=get_bytes(f'{SOURCE}/checkpoints/chunk_{group:03d}.zip')
    assert remote is not None, f'Missing original checkpoint group {group}'
    with zipfile.ZipFile(io.BytesIO(remote)) as archive:
        old_config=json.loads(archive.read('config.json'))
        assert old_config['prompt_hash']==old_manifest['prompt_hash']
        assert old_config['layers']==LAYERS
        for b in range(group*4,min((group+1)*4,old_nbatch)):
            with np.load(io.BytesIO(archive.read(f'batch_{b:03d}.npz')),allow_pickle=False) as z:
                assert str(z['prompt_hash'].item())==old_manifest['prompt_hash']
                assert int(z['start'])==b*PREV_BATCH
                assert int(z['stop'])==min((b+1)*PREV_BATCH,OLD_N)
                v=z['features'].copy()
                assert v.shape==(min((b+1)*PREV_BATCH,OLD_N)-b*PREV_BATCH,len(LAYERS),896)
                OLD_FEATURES.append(v)
OLD_FEATURES=np.concatenate(OLD_FEATURES)
assert OLD_FEATURES.shape==(OLD_N,3,896)
OLD_INDEX={(r.concept,r.split,int(r.family)):i for i,r in enumerate(old_stim.itertuples())}
assert len(OLD_INDEX)==len(old_stim)
def old_train(name,li):
    return np.stack([OLD_FEATURES[OLD_INDEX[(name,'train',k)],li] for k in range(8)])
assert all((name,'train',k) in OLD_INDEX for name in NAMES for k in range(8))
print('Frozen original training features:',OLD_FEATURES.shape)


## 4. Extract new Qwen hidden states — resumable checkpoints

The evaluation input uses the *same user-message format* and final assistant-start position as the previous run. Only the new fact texts differ. Checkpoints are immutable and validated by the **exact prompt hash**. When resuming, existing complete remote bundles take precedence over a local re-encoding to avoid ZIP-timestamp conflicts.

In [ ]:
INPUT_BYTES=NEW.to_csv(index=False).encode()
PROMPT_HASH=hashlib.sha256(INPUT_BYTES).hexdigest()
CONFIG={'run_id':RUN_ID,'source_run':SOURCE,'old_prompt_hash':old_manifest['prompt_hash'],
        'fresh_prompt_hash':PROMPT_HASH,'model':MODEL_ID,'revision':MODEL_REVISION,
        'layers':LAYERS,'position':POSITION,'batch_size':BATCH_SIZE,
        'n_eval_facts':len(NEW),'concepts':NAMES,'frozen_pairs_sha256':hashlib.sha256(PAIRS.to_csv(index=False).encode()).hexdigest()}
for fname in ['frozen_comparators.csv','fresh_eval_design.csv','fresh_eval_lexical_audit.csv']:
    save_bytes(f'{DEST}/{fname}',(OUT/fname).read_bytes(),'Freeze independent confirmation: '+fname)
save_bytes(f'{DEST}/config.json',json.dumps(CONFIG,sort_keys=True,indent=2).encode(),'Freeze confirmation run identity')
assert torch.cuda.is_available(),'Choose a Colab GPU runtime (T4 is sufficient).'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=torch.float32,attn_implementation='eager').to('cuda').eval()
assert model.config.num_hidden_layers==24 and model.config.hidden_size==896
@torch.inference_mode()
def extract(prompt):
    ids=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    h=model(input_ids=ids,output_hidden_states=True,use_cache=False).hidden_states
    return np.stack([h[layer][0,-1].float().cpu().numpy() for layer in LAYERS])

N=len(NEW); NB=math.ceil(N/BATCH_SIZE); NG=math.ceil(NB/GROUP_BATCHES)
def group_batches(g):return list(range(g*GROUP_BATCHES,min((g+1)*GROUP_BATCHES,NB)))
def batch_path(b):return CACHE/f'batch_{b:03d}.npz'
def remote_group(g):return f'{DEST}/checkpoints/chunk_{g:03d}.zip'
def verify_batch(b,raw):
    with np.load(io.BytesIO(raw),allow_pickle=False) as z:
        assert str(z['prompt_hash'].item())==PROMPT_HASH
        assert int(z['start'])==b*BATCH_SIZE and int(z['stop'])==min((b+1)*BATCH_SIZE,N)
        x=z['features']
        assert x.shape==(min((b+1)*BATCH_SIZE,N)-b*BATCH_SIZE,3,896)
        assert np.isfinite(x).all()
def recover(g):
    blob=get_bytes(remote_group(g))
    if blob is None:return False
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        assert json.loads(z.read('config.json'))==CONFIG,'Changed configuration: use a new run ID'
        assert set(z.namelist())=={'config.json'}|{f'batch_{b:03d}.npz' for b in group_batches(g)}
        for b in group_batches(g):
            content=z.read(f'batch_{b:03d}.npz');verify_batch(b,content)
            batch_path(b).write_bytes(content)
    print('Recovered immutable remote group',g)
    return True
def publish(g):
    if recover(g):return True
    if not all(batch_path(b).exists() for b in group_batches(g)):return False
    buf=io.BytesIO()
    with zipfile.ZipFile(buf,'w',compression=zipfile.ZIP_DEFLATED) as z:
        z.writestr('config.json',json.dumps(CONFIG,sort_keys=True))
        for b in group_batches(g):
            data=batch_path(b).read_bytes();verify_batch(b,data)
            z.writestr(f'batch_{b:03d}.npz',data)
    save_bytes(remote_group(g),buf.getvalue(),f'Freeze confirmation checkpoint group {g}')
    return True
for g in range(NG):recover(g)
for b in tqdm(range(NB),desc='New Qwen evaluation hidden states'):
    if batch_path(b).exists():verify_batch(b,batch_path(b).read_bytes());continue
    begin=b*BATCH_SIZE;end=min((b+1)*BATCH_SIZE,N)
    x=np.stack([extract(prompt) for prompt in NEW.user_prompt.iloc[begin:end]])
    buf=io.BytesIO();np.savez_compressed(buf,prompt_hash=np.array(PROMPT_HASH),start=np.array(begin),stop=np.array(end),features=x)
    batch_path(b).write_bytes(buf.getvalue())
    if (b+1)%GROUP_BATCHES==0 or b+1==NB:publish(b//GROUP_BATCHES)
for g in range(NG):assert publish(g)
EVAL_FEATURES=np.concatenate([np.load(batch_path(b),allow_pickle=False)['features'] for b in range(NB)])
assert EVAL_FEATURES.shape==(N,3,896)
EVAL_IDX={(r.concept,int(r.family)):i for i,r in enumerate(NEW.itertuples())}
def fresh_eval(name,li):return np.stack([EVAL_FEATURES[EVAL_IDX[name,k],li] for k in range(8)])
print('New evaluation tensor:',EVAL_FEATURES.shape)


## 5. Refit frozen Probes on old training inputs only; evaluate **new** sentences

The Probes are re-trained using only the **old eight training facts of X and Y**. Confirmed measures are the fresh AUC, the lowest AUC after removing each paired training fact index, and additional L23 AUC. Previously used evaluation sentences **never** enter the model fit.

The original screening thresholds are carried forward without tuning: L24 ≥0.85, lowest L24 leave-one-training-fact AUC ≥0.80, L23 ≥0.75. We also compute a 200-draw class-balanced label-permutation test for every unique selected pair; one-sided tail values and Benjamini–Hochberg adjusted q-values are descriptive because the human validity of new stimuli still needs review.

In [ ]:
def probe_fit(a,b,labels=None):
    x=np.concatenate([a,b])
    yy=np.array([1]*len(a)+[0]*len(b)) if labels is None else labels
    ss=StandardScaler().fit(x)
    clf=LogisticRegression(C=.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(ss.transform(x),yy)
    return ss,clf

def score(ss,clf,a,b):
    xx=np.concatenate([a,b]); y=np.array([1]*len(a)+[0]*len(b))
    return float(roc_auc_score(y,clf.decision_function(ss.transform(xx))))

unique_pairs=sorted(set(tuple(sorted((r.target,r.comparator))) for r in PAIRS.itertuples()))
rows=[];perm_count=200
for a,b in tqdm(unique_pairs,desc='Frozen X/Y Probe confirmation'):
    row={'pair':'__'.join((a,b)),'a':a,'b':b}
    for li,layer in enumerate(LAYERS):
        ta,tb=old_train(a,li),old_train(b,li)
        ea,eb=fresh_eval(a,li),fresh_eval(b,li)
        ss,clf=probe_fit(ta,tb)
        row[f'L{layer}_auc']=score(ss,clf,ea,eb)
        if layer==24:
            low=[]
            for omit in range(8):
                selected=[i for i in range(8) if i!=omit]
                ss2,c2=probe_fit(ta[selected],tb[selected]);low.append(score(ss2,c2,ea,eb))
            row['L24_loo_min']=min(low)
            row['L24_loo_mean']=float(np.mean(low))
            xx=np.concatenate([ta,tb]);ye=np.array([1]*8+[0]*8)
            sc=StandardScaler().fit(xx)
            zx=sc.transform(xx);ze=sc.transform(np.concatenate([ea,eb]))
            null=[]
            rng=np.random.default_rng(int(hashlib.sha256((row['pair']+str(SEED)).encode()).hexdigest()[:8],16))
            for _ in range(perm_count):
                labels=np.array([1]*8+[0]*8);rng.shuffle(labels)
                cc=LogisticRegression(C=.1,solver='liblinear',max_iter=1000,random_state=SEED).fit(zx,labels)
                null.append(float(roc_auc_score(ye,cc.decision_function(ze))))
            row['shuffle_tail_plus_one']=(1+sum(v>=row['L24_auc']-1e-12 for v in null))/(perm_count+1)
            row['shuffle_mean_auc']=float(np.mean(null))
    rows.append(row)
RESULT=pd.DataFrame(rows)
RESULT['threshold_pass']=(RESULT.L24_auc>=.85)&(RESULT.L24_loo_min>=.80)&(RESULT.L23_auc>=.75)
RESULT['shuffle_pass']=RESULT.shuffle_tail_plus_one<=.05
RESULT['confirmation_pass_unadjusted']=RESULT.threshold_pass&RESULT.shuffle_pass
RESULT['shuffle_fdr_q']=multipletests(RESULT.shuffle_tail_plus_one,method='fdr_bh')[1]
RESULT['confirmation_pass_fdr']=RESULT.threshold_pass&(RESULT.shuffle_fdr_q<=.05)
RESULT.to_csv(OUT/'frozen_pair_confirmation.csv',index=False)
print('Unique contrasts:',len(RESULT),'new-text threshold passes:',int(RESULT.threshold_pass.sum()),'plus shuffle:',int(RESULT.confirmation_pass_unadjusted.sum()))


## 6. Count the replicated target concepts and compare with old results

All eligible Ys and distance bands were selected from the old run **before** inspecting any new-data results. For each target, require >=2 confirmed distinct Ys in >=2 old frozen distance bands; also report a more conservative FDR-adjusted count. This is a **holdout confirmation** of a preselected shortlist, not a claim about 100 newly validated concepts.

In [ ]:
PAIRS=PAIRS.copy()
PAIRS['pair']=PAIRS.apply(lambda r:'__'.join(sorted((r.target,r.comparator))),axis=1)
JOIN=PAIRS.merge(RESULT,on='pair',how='left',validate='many_to_one')
assert len(JOIN)==len(PAIRS)
JOIN.to_csv(OUT/'directed_confirmation_by_distance.csv',index=False)
summary=[]
for x,g in JOIN.groupby('target',sort=False):
    p=g[g.confirmation_pass_unadjusted]
    q=g[g.confirmation_pass_fdr]
    summary.append({'target':x,'tier':g.tier.iloc[0],
       'frozen_comparisons':len(g),
       'n_confirmed_unadjusted':int(p.comparator.nunique()),
       'n_confirmed_bands_unadjusted':int(p.band.nunique()),
       'replicated_two_bands_unadjusted':bool(p.comparator.nunique()>=2 and p.band.nunique()>=2),
       'n_confirmed_fdr':int(q.comparator.nunique()),
       'n_confirmed_bands_fdr':int(q.band.nunique()),
       'replicated_two_bands_fdr':bool(q.comparator.nunique()>=2 and q.band.nunique()>=2),
       'confirmed_ys_unadjusted':', '.join(p.comparator.tolist()),
       'mean_L24_new_auc':float(g.L24_auc.mean()),
       'near_auc':float(g.loc[g.band=='near','L24_auc'].mean()) if (g.band=='near').any() else None,
       'medium_auc':float(g.loc[g.band=='medium','L24_auc'].mean()) if (g.band=='medium').any() else None,
       'far_auc':float(g.loc[g.band=='far','L24_auc'].mean()) if (g.band=='far').any() else None})
TARGET_REPORT=pd.DataFrame(summary).sort_values(['replicated_two_bands_unadjusted','n_confirmed_unadjusted','target'],ascending=[False,False,True])
TARGET_REPORT.to_csv(OUT/'target_confirmation.csv',index=False)
BANDS=JOIN.groupby('band',as_index=False).agg(n=('L24_auc','size'),mean_auc=('L24_auc','mean'),median_auc=('L24_auc','median'),confirm_unadjusted=('confirmation_pass_unadjusted','sum'),confirm_fdr=('confirmation_pass_fdr','sum'),mean_distance=('cosine_distance','mean'))
BANDS.to_csv(OUT/'distance_band_confirmation.csv',index=False)
print('Targets passing independent facts in >=2 bands (unadjusted):',int(TARGET_REPORT.replicated_two_bands_unadjusted.sum()),'/16')
print('Targets passing BH/FDR sensitivity:',int(TARGET_REPORT.replicated_two_bands_fdr.sum()),'/16')
print(TARGET_REPORT.to_string(index=False))
print(BANDS.to_string(index=False))
fig,ax=plt.subplots(figsize=(9,4.8))
order=TARGET_REPORT.target.to_list()
values=TARGET_REPORT.n_confirmed_unadjusted.to_list()
ax.bar(order,values,color='#39768d')
ax.set_ylabel('Confirmed distinct comparators Y')
ax.set_title('Preselected 16 targets: independent new-fact confirmation')
ax.tick_params(axis='x',rotation=65)
fig.tight_layout();fig.savefig(OUT/'confirmed_targets.png',dpi=180);plt.show()


## 7. Manifest, provenance, full ZIP, GitHub result publishing

No actual results are declared until this final cell runs. Since this is confirmation of a *preselected* shortlist rather than independent human validation, interpret the result as a new-text replication with content-review caveats. Publish reproducibility files and results to a **new immutable GitHub path**.

In [ ]:
manifest={**CONFIG,'status':'COMPLETED_NEW_TEXT_CONFIRMATION',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'gpu':torch.cuda.get_device_name(0),
    'original_training_examples_used':8,
    'original_screening_examples_used_for_training':0,
    'new_evaluation_facts_per_concept':8,
    'new_evaluation_concept_count':len(NAMES),
    'new_qwen_prompts':N,
    'primary_targets':PRIMARY,
    'secondary_targets':SECONDARY,
    'frozen_directed_pairs':len(PAIRS),
    'unique_pair_contrasts':len(RESULT),
    'shuffle_draws_per_unique_pair':perm_count,
    'unadjusted_two_band_targets':int(TARGET_REPORT.replicated_two_bands_unadjusted.sum()),
    'fdr_two_band_targets':int(TARGET_REPORT.replicated_two_bands_fdr.sum()),
    'limitations':['New factual descriptions were AI-authored, not independently human annotated.',
                   'Facts require domain validity review before confirmatory interpretation.',
                   'The original 16 target concepts and comparator Ys were preselected using previous evaluation AUCs.',
                   'Permutation tails are a screening diagnostic, not a calibrated test of concept activation.',
                   'Reusing the old training vectors means this tests fresh generalization of an unchanged probe family.',
                   'AUC measures relative X-versus-Y identity discrimination and cannot establish suppression or thought.']}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
report=f"""# Independent new-description confirmation: Qwen Concept Probe

- **Frozen primary targets:** {len(PRIMARY)}; secondary near-misses: {len(SECONDARY)}.
- **Newly authored evaluation statements:** {len(NEW)} (8 per concept, {len(NAMES)} unique concepts).
- **Source:** previous frozen training vectors only; no old evaluation vectors in classifier fit.
- **Frozen directed comparisons:** {len(PAIRS)}; unique unordered pairs: {len(RESULT)}.
- **New-text threshold-only pairs:** {int(RESULT.threshold_pass.sum())}.
- **New-text threshold + shuffle-passing pairs:** {int(RESULT.confirmation_pass_unadjusted.sum())}.
- **New-text threshold + FDR-adjusted shuffle pairs:** {int(RESULT.confirmation_pass_fdr.sum())}.
- **Distinct targets replicating across >=2 Ys and >=2 semantic bands:** {int(TARGET_REPORT.replicated_two_bands_unadjusted.sum())}/16 (unadjusted), {int(TARGET_REPORT.replicated_two_bands_fdr.sum())}/16 (FDR-sensitivity).

## Target-by-target confirmation

{TARGET_REPORT.to_markdown(index=False)}

## Distance bands

{BANDS.to_markdown(index=False)}

**Interpretation:** AUC measures relative concept identity discrimination, not absolute activation or suppression. The newly written statements must be independently reviewed for factual validity, lack of duplicate semantic content, and target-label ambiguity. This is *not* a claim of independently human-verified confirmation.
"""
(OUT/'analysis_summary.md').write_text(report)
archive=OUT/'independent_concept_probe_confirmation_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file() and f!=archive:z.write(f,f.name)
for file in ['manifest.json','analysis_summary.md','target_confirmation.csv','frozen_pair_confirmation.csv','distance_band_confirmation.csv','directed_confirmation_by_distance.csv','confirmed_targets.png']:
    save_bytes(f'{DEST}/{file}',(OUT/file).read_bytes(),'Save independently authored text confirmation '+file)
save_bytes(f'{DEST}/{archive.name}',archive.read_bytes(),'Save complete independent confirmation result bundle')
print('RESULTS: https://github.com/'+REPO+'/tree/main/'+DEST)
